# Volve 15/9-F-4 Black Oil EOS 
# Step 1 — Composition & Critical Property Characterisation

**Well:** 15/9-F-4, Volve Field, North Sea  
**Sample:** Bottle 6103-MA (12.5 wt% OBM contamination corrected)  
**Source:** Reslab PVT Report 2008, Table 4.3.8 (corrected wellstream)  
**EOS:** Peng-Robinson 3-parameter (PR3) with Kesler-Lee correlations  
**Reservoir conditions:** T = 107°C, Pi = 332.8 bar, Pbp = 213.1 bar 

**Done in commercial sofware:** Reproduced in this notebook

In [15]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# section
def styled(df, title=None, precision=4):
    """Render a DataFrame as a styled HTML table."""
    styled_df = (
        df.style
        .format(precision=precision)
        .set_table_styles([
            {'selector': 'thead th',
             'props': [('background-color', '#2c3e50'),
                       ('color', 'white'),
                       ('font-weight', 'bold'),
                       ('text-align', 'center'),
                       ('padding', '8px 12px'),
                       ('font-size', '12px')]},
            {'selector': 'tbody td',
             'props': [('text-align', 'right'),
                       ('padding', '6px 12px'),
                       ('font-size', '12px')]},
            {'selector': 'tbody tr:nth-child(even)',
             'props': [('background-color', '#f2f2f2')]},
            {'selector': 'tbody tr:hover',
             'props': [('background-color', '#d6eaf8')]},
            {'selector': 'table',
             'props': [('border-collapse', 'collapse'),
                       ('width', '100%'),
                       ('box-shadow', '0 2px 4px rgba(0,0,0,0.1)')]},
            {'selector': 'th:first-child, td:first-child',
             'props': [('text-align', 'left'),
                       ('font-weight', 'bold')]},
        ])
        .set_caption(title or '')
    )
    # highlight pseudo-component rows
    def highlight_pseudo(row):
        if row.name in ['C7-C15', 'C16+']:
            return ['background-color: #eaf4fb; font-weight: bold'] * len(row)
        return [''] * len(row)
    styled_df = styled_df.apply(highlight_pseudo, axis=1)
    display(styled_df)

print('Styling helper loaded.')

Styling helper loaded.


# 1.1 Wellstream Composition (Table 4.3.8)

Two pseudo-components replace the single C7+ lump:  
- **C7-C15**: light end of plus fraction (C7 through C15)  
- **C16+**: heavy residue (C16 through C36+)  

MW and SG computed by weight-averaging SCN data from Table 4.3.7.

In [16]:
# section
# Component names
NAMES = ['N2', 'CO2', 'C1', 'C2', 'C3', 'iC4', 'nC4', 'iC5', 'nC5', 'C6',
         'C7-C15', 'C16+']
NC = len(NAMES)

# Mole fractions ZI (mol%) from Table 4.3.8 corrected wellstream
ZI_PCT = np.array([
    0.410,   # N2
    3.798,   # CO2
   39.901,   # C1
    6.070,   # C2
    5.447,   # C3
    0.771,   # iC4
    2.815,   # nC4
    1.050,   # iC5
    1.696,   # nC5
    2.341,   # C6
   19.767,   # C7-C15  (sum C7-C15 from Table 4.3.8)
   15.897,   # C16+    (sum C16-C36+ from Table 4.3.8)
])
ZI = ZI_PCT / 100.0

#check sum
#print(f'Sum ZI = {ZI_PCT.sum():.4f} mol%  (should be ~100)')

# Molecular weights
# Library components: standard values
# Pseudo-components: weight-average from Table 4.3.7 SCN data
MW = np.array([
    28.013,  # N2
    44.010,  # CO2
    16.043,  # C1
    30.070,  # C2
    44.097,  # C3
    58.124,  # iC4
    58.124,  # nC4
    72.151,  # iC5
    72.151,  # nC5
    86.178,  # C6
   150.000,  # C7-C15  
   480.000,  # C16+    
])

# Specific gravities (relative to water = 1.0) for pseudo-components
# Library components: not used directly (standard critical props used)
SG = np.array([
    np.nan,  # N2
    np.nan,  # CO2
    np.nan,  # C1
    np.nan,  # C2
    np.nan,  # C3
    np.nan,  # iC4
    np.nan,  # nC4
    np.nan,  # iC5
    np.nan,  # nC5
    np.nan,  # C6
    0.784,   # C7-C15  (weight-average from Table 4.3.7)
    0.968,   # C16+    (weight-average from Table 4.3.7)
])

# Weight fractions
mass = ZI * MW
WI = mass / mass.sum()

# Build display table
df_comp = pd.DataFrame({
    'ZI (mol%)': ZI_PCT,
    'WI (wt%)': WI * 100,
    'MW (g/mol)': MW,
    'SG': SG,
}, index=NAMES)

styled(df_comp,
       title='Table 1 — Wellstream Composition (Volve 15/9-F-4, corrected for OBM contamination)',
       precision=3)

,ZI (mol%),WI (wt%),MW (g/mol),SG
N2,0.410,0.092,28.013,nan
CO2,3.798,1.343,44.010,nan
C1,39.901,5.144,16.043,nan
C2,6.070,1.467,30.070,nan
C3,5.447,1.930,44.097,nan
iC4,0.771,0.360,58.124,nan
nC4,2.815,1.315,58.124,nan
iC5,1.050,0.609,72.151,nan
nC5,1.696,0.983,72.151,nan
C6,2.341,1.621,86.178,nan


# 1.2 Library Component Critical Properties (Kesler-Lee / KF)

Standard critical properties for N2, CO2 and C1-C6 from the PVTi KF library.  
These are fixed — not regressed.

In [17]:
# section
# Tc (K), Pc (bar), omega (acentric factor)
# Source: KF library, confirmed from CCE output

TC_LIB = np.array([
    126.20,  # N2
    304.20,  # CO2
    190.60,  # C1
    305.40,  # C2
    369.80,  # C3
    408.10,  # iC4
    425.20,  # nC4
    460.40,  # iC5
    469.60,  # nC5
    507.40,  # C6
    np.nan,  # C7-C15 → from KF correlation
    np.nan,  # C16+   → from KF correlation
])  # K

PC_LIB = np.array([
    33.94,   # N2
    73.82,   # CO2
    46.10,   # C1
    48.84,   # C2
    42.46,   # C3
    36.48,   # iC4
    38.00,   # nC4
    33.84,   # iC5
    33.74,   # nC5
    29.69,   # C6
    np.nan,  # C7-C15
    np.nan,  # C16+
])  # bar

OMEGA_LIB = np.array([
    0.0400,  # N2
    0.2250,  # CO2
    0.0130,  # C1
    0.0986,  # C2
    0.1524,  # C3
    0.1760,  # iC4
    0.2010,  # nC4
    0.2280,  # iC5
    0.2510,  # nC5
    0.2960,  # C6
    np.nan,  # C7-C15
    np.nan,  # C16+
])

print('Library critical properties defined for C1-C10.')

Library critical properties defined for C1-C10.


# 1.3 Kesler-Lee Correlations for Pseudo-Component Critical Properties

commercial software uses the **Kesler-Lee (1976)** correlations to calculate Tc, Pc and omega  
for characterised plus fractions from MW and SG.

$$T_c = 341.7 + 811.1 \cdot SG + (0.4244 + 0.1174 \cdot SG) \cdot T_b + \frac{(0.4669 - 3.2623 \cdot SG) \times 10^5}{T_b}$$

where $T_b$ is the mean average boiling point (K), also estimated from MW and SG.

In [18]:
# section
# Watson Kw → Tb, then Kesler-Lee for Tc and Pc, Lee-Kesler for omega
# All match pre-regression characterisation exactly
# Ref: Whitson & Brule (2000) Eqs. 5.52, 5.56; Kesler & Lee (1976)

def kesler_lee_Tb(MW, SG):
    """Tb (K) via Watson characterisation factor KF library."""
    Kw   = 4.5579 * (MW ** 0.15178) * (SG ** (-0.84573))
    Tb_R = (Kw * SG) ** 3
    return Tb_R * 5/9

def kesler_lee_Tc(Tb_K, SG):
    """Tc (K) — Kesler-Lee (1976), Tb in Rankine. Whitson Eq. 5.52."""
    Tb_R = Tb_K * 9/5
    Tc_R = (341.7 + 811.1*SG
            + (0.4244 + 0.1174*SG)*Tb_R
            + (0.4669 - 3.2623*SG)*1e5 / Tb_R)
    return Tc_R * 5/9

def kesler_lee_Pc(Tb_K, SG, MW):
    """Pc (bar) — Kesler-Lee (1976), Tb in Rankine. Whitson Eq. 5.56."""
    Tb_R = Tb_K * 9/5
    ln_Pc = (8.3634 - 0.0566/SG
             - (0.24244 + 2.2898/SG + 0.11857/SG**2)*1e-3*Tb_R
             + (1.4685  + 3.648/SG  + 0.47227/SG**2)*1e-7*Tb_R**2
             - (0.42019 + 1.6977/SG**2)*1e-10*Tb_R**3)
    return np.exp(ln_Pc) * 0.0689476

def lee_kesler_omega(Tb_K, Tc_K, Pc_bar, SG):
    """
    Acentric factor — Lee-Kesler (1975/1976).
    Two forms depending on reduced boiling temperature:
      Tbr <= 0.8: vapor-pressure definition form (lighter fractions)
      Tbr >  0.8: Watson-K petroleum-fraction form (heavy fractions)
    Ref: Kesler & Lee (1976), Hydrocarbon Processing.
    """
    Tb_R    = Tb_K * 9/5
    Tc_R    = Tc_K * 9/5
    Tbr     = Tb_R / Tc_R
    Kw      = Tb_R**(1/3) / SG
    Pc_psia = Pc_bar / 0.0689476

    if Tbr <= 0.8:
        # Vapor-pressure definition form
        num = (-np.log(Pc_psia/14.7) - 5.92714 + 6.09648/Tbr
               + 1.28862*np.log(Tbr) - 0.169347*Tbr**6)
        den = (15.2518 - 15.6875/Tbr
               - 13.4721*np.log(Tbr) + 0.43577*Tbr**6)
        return num / den
    else:
        # Watson-K petroleum-fraction form
        return (-7.904 + 0.1352*Kw - 0.007465*Kw**2
                + 8.359*Tbr + (1.408 - 0.01063*Kw)/Tbr)

# section
pseudo_idx   = [10, 11]
pseudo_names = ['C7-C15', 'C16+']

results = []
for idx, name in zip(pseudo_idx, pseudo_names):
    mw = MW[idx]
    sg = SG[idx]
    Tb = kesler_lee_Tb(mw, sg)
    Tc = kesler_lee_Tc(Tb, sg)
    Pc = kesler_lee_Pc(Tb, sg, mw)
    om = lee_kesler_omega(Tb, Tc, Pc, sg)
    results.append({
        'Component': name,
        'MW': mw, 'SG': sg,
        'Tb (K)': round(Tb, 2),
        'Tc (K)': round(Tc, 2),
        'Pc (bar)': round(Pc, 3),
        'omega': round(om, 4),
    })
    TC_LIB[idx]    = Tc
    PC_LIB[idx]    = Pc
    OMEGA_LIB[idx] = om

df_kl = pd.DataFrame(results).set_index('Component')
styled(df_kl,
       title='Table 2 — Kesler-Lee Critical Properties for Pseudo-Components '
             '(pre-regression, matches commerical software characterisation)',
       precision=3)

,MW,SG,Tb (K),Tc (K),Pc (bar),omega
Component,,,,,,
C7-C15,150.000,0.784,460.230,640.580,22.841,0.486
C16+,480.000,0.968,861.710,993.280,6.941,1.373


In [19]:
# section
# Library components (N2, CO2, C1-C6)
# Pseudo-components (C7-C15, C16+): Kesler-Lee/Lee-Kesler correlations
# All verified against commercial software pre-regression component properties screen

TB_LIB = np.array([
   -195.75 + 273.15,   # N2
    -78.45 + 273.15,   # CO2
   -161.55 + 273.15,   # C1
    -88.55 + 273.15,   # C2
    -42.05 + 273.15,   # C3
    -11.85 + 273.15,   # iC4
     -0.45 + 273.15,   # nC4
     27.85 + 273.15,   # iC5
     36.05 + 273.15,   # nC5
     63.90 + 273.15,   # C6
    187.08 + 273.15,   # C7-C15 — Watson Kw
    588.56 + 273.15,   # C16+   — Watson Kw
])  # K

VCRIT_LIB = np.array([
    0.090, 0.094, 0.098, 0.148, 0.200,
    0.263, 0.255, 0.308, 0.311, 0.351,
    0.59434,   # C7-C15
    1.85754,   # C16+
])  # m³/kg-mol

df_crit = pd.DataFrame({
    'MW (g/mol)':      MW,
    'Tb (K)':          TB_LIB,
    'Tc (K)':          TC_LIB,
    'Pc (bar)':        PC_LIB,
    'omega':           OMEGA_LIB,
    'Vcrit (m³/kmol)': VCRIT_LIB,
}, index=NAMES)

styled(df_crit,
       title='Table 3 — Full Critical Property Set '
             '(KF Library + Kesler-Lee for pseudo-components)',
       precision=3)

,MW (g/mol),Tb (K),Tc (K),Pc (bar),omega,Vcrit (m³/kmol)
N2,28.013,77.400,126.200,33.940,0.040,0.090
CO2,44.010,194.700,304.200,73.820,0.225,0.094
C1,16.043,111.600,190.600,46.100,0.013,0.098
C2,30.070,184.600,305.400,48.840,0.099,0.148
C3,44.097,231.100,369.800,42.460,0.152,0.200
iC4,58.124,261.300,408.100,36.480,0.176,0.263
nC4,58.124,272.700,425.200,38.000,0.201,0.255
iC5,72.151,301.000,460.400,33.840,0.228,0.308
nC5,72.151,309.200,469.600,33.740,0.251,0.311
C6,86.178,337.050,507.400,29.690,0.296,0.351


# 1.4 Binary Interaction Parameters (KF Library)

KF BIPs non-zero only for N2/CO2 pairs and C1/C6+ pairs.

In [20]:
# section
# KF library BIPs confirmed 
# Non-zero only for:
# N2/CO2 pair: -0.012
# N2, CO2 with all HC: 0.1
# C1/C6: 0.0279, C1/C7-C15: 0.04176, C1/C16+: 0.06752
# C2,C3 with C6,C7-C15,C16+: 0.01

BIP = np.zeros((NC, NC))

# N2 / CO2
BIP[0,1] = BIP[1,0] = -0.012

# N2 with all HC (C1-C16+)
for j in range(2, NC):
    BIP[0,j] = BIP[j,0] = 0.1

# CO2 with all HC (C1-C16+)
for j in range(2, NC):
    BIP[1,j] = BIP[j,1] = 0.1

# C1 / C6
BIP[2,9]  = BIP[9,2]  = 0.0279
# C1 / C7-C15
BIP[2,10] = BIP[10,2] = 0.04176
# C1 / C16+
BIP[2,11] = BIP[11,2] = 0.06752

# C2 / C6, C7-C15, C16+
BIP[3,9]  = BIP[9,3]  = 0.01
BIP[3,10] = BIP[10,3] = 0.01
BIP[3,11] = BIP[11,3] = 0.01

# C3 / C6, C7-C15, C16+
BIP[4,9]  = BIP[9,4]  = 0.01
BIP[4,10] = BIP[10,4] = 0.01
BIP[4,11] = BIP[11,4] = 0.01

# section
df_bip = pd.DataFrame(BIP, index=NAMES, columns=NAMES)

def highlight_nonzero(val):
    if isinstance(val, float) and abs(val) > 1e-6:
        return 'background-color: #d5f5e3; font-weight: bold'
    return 'color: #aaaaaa'

(
    df_bip.style
    .format('{:.5f}')
    .map(highlight_nonzero)
    .set_caption('Table 4 — Binary Interaction Parameters (KF Library defaults, PR EOS)')
    .set_table_styles([
        {'selector': 'thead th',
         'props': [('background-color','#2c3e50'),('color','white'),
                   ('font-size','11px'),('padding','5px 8px')]},
        {'selector': 'tbody td',
         'props': [('font-size','11px'),('padding','4px 8px'),
                   ('text-align','right')]},
        {'selector': 'th:first-child, td:first-child',
         'props': [('background-color','#2c3e50'),('color','white'),
                   ('font-weight','bold'),('font-size','11px')]},
    ])
)

,N2,CO2,C1,C2,C3,iC4,nC4,iC5,nC5,C6,C7-C15,C16+
N2,0.00000,-0.01200,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000
CO2,-0.01200,0.00000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000,0.10000
C1,0.10000,0.10000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.02790,0.04176,0.06752
C2,0.10000,0.10000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.01000,0.01000,0.01000
C3,0.10000,0.10000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.01000,0.01000,0.01000
iC4,0.10000,0.10000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
nC4,0.10000,0.10000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
iC5,0.10000,0.10000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
nC5,0.10000,0.10000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
C6,0.10000,0.10000,0.02790,0.01000,0.01000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000


In [21]:
# section
print('=' * 60)
print('STEP 1 COMPLETE — EOS Setup Summary')
print('=' * 60)
print(f'\nComponents:        {NC}')
print(f'Sum ZI:            {ZI_PCT.sum():.4f} mol%')
print(f'\nPseudo-component critical properties (Kesler-Lee):')
for name in ['C7-C15', 'C16+']:
    idx = NAMES.index(name)
    print(f'  {name:8s}: Tb={TB_LIB[idx]:.2f}K  Tc={TC_LIB[idx]:.2f}K  '
          f'Pc={PC_LIB[idx]:.3f} bar  omega={OMEGA_LIB[idx]:.4f}')
print(f'\nCommercial software pre-regression reference:')
print(f'  C7-C15 : Tc=640.54K  Pc=22.835 bar  omega=0.4867')
print(f'  C16+   : Tc=993.22K  Pc=6.939 bar   omega=1.3738')
print(f'\nBIP matrix: {(BIP != 0).sum()//2} non-zero pairs')
print(f'EOS: PR3 (Peng-Robinson 3-parameter) with PRCORR')
print(f'T reservoir: 107°C = 380.15K')
print(f'Target Pbp: 213.1 bar')


STEP 1 COMPLETE — EOS Setup Summary

Components:        12
Sum ZI:            99.9630 mol%

Pseudo-component critical properties (Kesler-Lee):
  C7-C15  : Tb=460.23K  Tc=640.58K  Pc=22.841 bar  omega=0.4863
  C16+    : Tb=861.71K  Tc=993.28K  Pc=6.941 bar  omega=1.3735

Commercial software pre-regression reference:
  C7-C15 : Tc=640.54K  Pc=22.835 bar  omega=0.4867
  C16+   : Tc=993.22K  Pc=6.939 bar   omega=1.3738

BIP matrix: 30 non-zero pairs
EOS: PR3 (Peng-Robinson 3-parameter) with PRCORR
T reservoir: 107°C = 380.15K
Target Pbp: 213.1 bar
